Practical: Write a CUDA Program for :
1. Addition of two large vectors
2. Matrix Multiplication using CUDA C


In [ ]:
!pip install git+https://github.com/afnan47/cuda.git

  Cloning https://github.com/afnan47/cuda.git to /tmp/pip-req-build-o6d7z9zr
  Running command git clone --filter=blob:none --quiet https://github.com/afnan47/cuda.git /tmp/pip-req-build-o6d7z9zr
  Resolved https://github.com/afnan47/cuda.git to commit aac710a35f52bb78ab34d2e52517237941399eff
  Preparing metadata (setup.py) ... done
  Created wheel for NVCCPlugin: filename=NVCCPlugin-0.0.2-py3-none-any.whl size=4290 sha256=b57180f9ed2db1cf8026e798a1e91ef2d6e468f85d6ed67076897d11ac2015d3
  Stored in directory: /tmp/pip-ephem-wheel-cache-_go8po_p/wheels/e8/cf/c3/c90ca0d0bba7969f9b8670f5624f76d097123d656355c77053
Successfully built NVCCPlugin


In [ ]:
!nvidia-smi

Sun May  3 09:15:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
!pip install nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpgh12jxub".


In [ ]:
%%cuda
#include <iostream>
#include <cuda_runtime.h>
using namespace std;

__global__ void vectorAdd(int *a,int *b,int *c,int n){
    int i=blockIdx.x*blockDim.x+threadIdx.x;
    if(i<n) c[i]=a[i]+b[i];
}

__global__ void matrixMul(int *a,int *b,int *c,int n){
    int r=threadIdx.y+blockDim.y*blockIdx.y;
    int col=threadIdx.x+blockDim.x*blockIdx.x;
    if(r<n && col<n){
        int sum=0;
        for(int k=0;k<n;k++)
            sum+=a[r*n+k]*b[k*n+col];
        c[r*n+col]=sum;
    }
}

int main(){

    // -------- VECTOR ADDITION --------
    int n=1024;
    int *a=new int[n], *b=new int[n], *c=new int[n], *d=new int[n];

    for(int i=0;i<n;i++){
        a[i]=i;
        b[i]=i*2;
        d[i]=a[i]+b[i];
    }

    int *da,*db,*dc;
    cudaMalloc(&da,n*sizeof(int));
    cudaMalloc(&db,n*sizeof(int));
    cudaMalloc(&dc,n*sizeof(int));

    cudaMemcpy(da,a,n*sizeof(int),cudaMemcpyHostToDevice);
    cudaMemcpy(db,b,n*sizeof(int),cudaMemcpyHostToDevice);

    cudaEvent_t s1,e1;
    float t1;
    cudaEventCreate(&s1); cudaEventCreate(&e1);

    // Warm-up (fix high first time)
    vectorAdd<<<(n+255)/256,256>>>(da,db,dc,n);
    cudaDeviceSynchronize();

    // Actual timing
    cudaEventRecord(s1);
    vectorAdd<<<(n+255)/256,256>>>(da,db,dc,n);
    cudaEventRecord(e1);
    cudaEventSynchronize(e1);
    cudaEventElapsedTime(&t1,s1,e1);

    cudaMemcpy(c,dc,n*sizeof(int),cudaMemcpyDeviceToHost);

    int err1=0;
    for(int i=0;i<n;i++) err1+=d[i]-c[i];

    cout<<"1."<<endl;
    cout<<"Error : "<<err1<<endl;
    cout<<"Time Elapsed: "<<t1<<endl;


    // -------- MATRIX MULTIPLICATION --------
    int m=3;
    int *A=new int[m*m], *B=new int[m*m], *C=new int[m*m], *D=new int[m*m];

    for(int i=0;i<m*m;i++){
        A[i]=2;
        B[i]=1;
    }

    int *dA,*dB,*dC;
    cudaMalloc(&dA,m*m*sizeof(int));
    cudaMalloc(&dB,m*m*sizeof(int));
    cudaMalloc(&dC,m*m*sizeof(int));

    cudaMemcpy(dA,A,m*m*sizeof(int),cudaMemcpyHostToDevice);
    cudaMemcpy(dB,B,m*m*sizeof(int),cudaMemcpyHostToDevice);

    cudaEvent_t s2,e2;
    float t2;
    cudaEventCreate(&s2); cudaEventCreate(&e2);

    dim3 threads(m,m);

    // Timing
    cudaEventRecord(s2);
    matrixMul<<<1,threads>>>(dA,dB,dC,m);
    cudaEventRecord(e2);
    cudaEventSynchronize(e2);
    cudaEventElapsedTime(&t2,s2,e2);

    cudaMemcpy(C,dC,m*m*sizeof(int),cudaMemcpyDeviceToHost);

    // CPU result
    for(int i=0;i<m;i++)
        for(int j=0;j<m;j++){
            int sum=0;
            for(int k=0;k<m;k++)
                sum+=A[i*m+k]*B[k*m+j];
            D[i*m+j]=sum;
        }

    int err2=0;
    for(int i=0;i<m*m;i++) err2+=D[i]-C[i];

    cout<<"2."<<endl;
    cout<<"Error : "<<err2<<endl;
    cout<<"Time Elapsed: "<<t2<<endl;

    return 0;
}

1.
Error : 0
Time Elapsed: 0.008192
2.
Error : 0
Time Elapsed: 0.030656



In [10]:
!jupyter nbconvert --to html "/content/Programs_using_CUDA_C.ipynb"

[NbConvertApp] Converting notebook /content/Programs_using_CUDA_C.ipynb to html
[NbConvertApp] Writing 309379 bytes to /content/Programs_using_CUDA_C.html
